In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

In [31]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Exploring games data

In [32]:
clean_games_fp = Path(config['games_folder'] + config['games_clean_fp'])

games_df = pd.read_json(clean_games_fp, orient='records')
games_df['updated_at'] = pd.to_datetime(games_df['updated_at'], errors='coerce')
games_df['first_release_date'] = pd.to_datetime(games_df['first_release_date'], errors='coerce')

games_df.head()

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,Apotris,-1.0,2026-05-01 15:42:21,2022-04-29,227679,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,Farm to Table,-1.0,2026-05-01 15:38:24,2026-05-09,356719,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,Blast Brawl 2: Bloody Boogaloo,90.0,2026-05-01 15:36:00,2016-10-25,33414,0,1,0,1,1,...,0,0,0,1,0,0,0,0,0,0
3,Speed Rivals,-1.0,2026-05-01 15:35:15,2026-04-30,348939,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,Anno 117: Pax Romana,-1.0,2026-05-01 15:31:55,2025-11-13,305246,0,1,0,1,1,...,0,0,0,0,1,0,0,0,0,0


In [33]:
games_df.head()

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,Apotris,-1.0,2026-05-01 15:42:21,2022-04-29,227679,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,Farm to Table,-1.0,2026-05-01 15:38:24,2026-05-09,356719,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,Blast Brawl 2: Bloody Boogaloo,90.0,2026-05-01 15:36:00,2016-10-25,33414,0,1,0,1,1,...,0,0,0,1,0,0,0,0,0,0
3,Speed Rivals,-1.0,2026-05-01 15:35:15,2026-04-30,348939,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,Anno 117: Pax Romana,-1.0,2026-05-01 15:31:55,2025-11-13,305246,0,1,0,1,1,...,0,0,0,0,1,0,0,0,0,0


In [34]:
#Checking to make sure the id uniquely identifies each row
print("No duplicate IDS") if len(games_df['id'].unique()) == len(games_df) else print("Duplicate IDs found")

No duplicate IDS


In [35]:
#Checking to see if duplicate game names exist
print("No duplicate game names") if len(games_df['name'].unique()) == len(games_df) else print("Duplicate game names found")

Duplicate game names found


In [36]:
#Get all rows with duplicate game names
duplicate_names_df = games_df[games_df.duplicated(subset=['name'], keep=False)].sort_values('name')
duplicate_names_df.head(6)

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
17029,10-Pin Bowling,-1.000000,2024-11-14 09:33:09,1984-12-31,153453,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
17030,10-Pin Bowling,-1.000000,2024-11-14 09:27:27,1999-08-01,92273,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
7366,15 Minutes,-1.000000,2026-04-09 19:38:22,2026-01-05,395433,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
12963,15 Minutes,-1.000000,2026-02-02 16:52:35,2025-10-23,355071,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5948,1942,61.392350,2026-04-18 08:37:59,1985-12-11,272544,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
6791,1942,67.742592,2026-04-13 08:03:19,1984-12-01,6075,0,1,0,1,1,...,0,0,0,0,0,1,0,0,0,0


In [37]:
#Find columns where duplicates differ (excluding name, rating, updated_at, id)
exclude_cols = {'name', 'rating', 'updated_at', 'id'}
cols_to_check = [col for col in duplicate_names_df.columns if col not in exclude_cols]

i = 0
for game_name in duplicate_names_df['name'].unique():
    game_group = duplicate_names_df[duplicate_names_df['name'] == game_name]
    differing_cols = []
    for col in cols_to_check:
        if len(game_group[col].unique()) > 1:
            differing_cols.append(col)
    if'first_release_date' not in differing_cols:  # Print every 10th game to avoid too much output
        #print(f"{game_name}: {differing_cols}")
        pass
    i+=1

There are many duplicates but they may functionally differ so we'll keep them for now

## Conflicts in game modes columns

There are several game mode columns that seem to imply multiplayer functionality (such as Co-operative or Massively Multiplayer Online), but we should check if they actually do inherently imply multiplayer compatibility. If not, we should make sure we understand what the column is referring to.

In [38]:
coop_without_multiplayer = games_df.groupby(['game_modes_Co-operative', 'game_modes_Multiplayer'])[['name', 'first_release_date']].agg(lambda x: x.sample(1))
coop_without_multiplayer

name  \
game_modes_Co-operative game_modes_Multiplayer                                             
0                       0                                              Detective Pikachu   
                        1                       Marvel vs. Capcom: Clash of Super Heroes   
1                       1                                             Sonic Robo Blast 2   

                                                    first_release_date  
game_modes_Co-operative game_modes_Multiplayer                          
0                       0                      1999-11-27 00:00:00.000  
                        1                      2019-07-23 00:00:00.000  
1                       1                      1677-09-21 00:12:43.145

For co-op, it seems like an error in entry; games with coop should have a 1 in the multiplayer column.

In [39]:
br_without_multiplayer = games_df.groupby(['game_modes_Battle Royale', 'game_modes_Multiplayer'])[['name', 'first_release_date']].agg(lambda x: x.sample(1))
br_without_multiplayer

name  \
game_modes_Battle Royale game_modes_Multiplayer                                                
0                        0                                     Jewel Master: Cradle of Egypt   
                         1                                                       Sheep Sweep   
1                        0                       Fall Guys: Season 4 - Creative Construction   
                         1                                                  Life and Death 2   

                                                     first_release_date  
game_modes_Battle Royale game_modes_Multiplayer                          
0                        0                      2025-03-13 00:00:00.000  
                         1                      2000-12-21 00:00:00.000  
1                        0                      1677-09-21 00:12:43.145  
                         1                      2022-08-29 00:00:00.000

Same for MMO

# Exploring multiplayer modes data

In [40]:
clean_modes_fp = Path(config['multiplayer_modes_folder'] + config['multiplayer_modes_clean_fp'])

modes_df = pd.read_json(clean_modes_fp, orient='records')

print("Contains " + str(len(modes_df)) + " rows")
modes_df.head()

Contains 23872 rows


,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen
0,9953,92273,False,False,False,0,2,False,0,0,Game Boy Color,False
1,1832,7153,True,True,True,2,0,False,0,0,Xbox 360,False
2,7987,57887,False,False,True,2,2,False,0,0,Xbox One,False
3,7,46076,False,False,False,0,30,False,0,0,PC (Microsoft Windows),False
4,10207,31256,False,False,False,0,0,False,0,0,Web browser,False


In [41]:
modes_df.isna().mean().sort_values(ascending=False)

id                0.0
game              0.0
dropin            0.0
campaigncoop      0.0
offlinecoop       0.0
offlinecoopmax    0.0
offlinemax        0.0
onlinecoop        0.0
onlinecoopmax     0.0
onlinemax         0.0
platform          0.0
splitscreen       0.0
dtype: float64

-1 as a fill value works fine for all of these

In [42]:
def isunique(df, subset):
    return len(df[subset].unique()) == len(df)

print("No duplicate IDS") if isunique(modes_df, 'id') else print("Duplicate IDs found")
print("No duplicate game ids") if isunique(modes_df, 'game') else print("Duplicate game ids found")

No duplicate IDS
Duplicate game ids found


In [43]:
duplicate_game_ids_df = modes_df[modes_df.duplicated(subset=['game'], keep=False)].sort_values('game')
duplicate_game_ids_df = duplicate_game_ids_df.merge(games_df[['id', 'name']], left_on='game', right_on='id', how='left', suffixes=('_mode', '_game'))
duplicate_game_ids_df.head(10)

,id_mode,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,id_game,name
0,11595,72,False,False,True,2,0,True,2,0,Linux,True,72.0,Portal 2
1,11592,72,False,False,True,2,0,True,2,0,PlayStation 3,True,72.0,Portal 2
2,11594,72,False,False,True,2,0,True,2,0,Mac,True,72.0,Portal 2
3,11593,72,False,False,True,2,0,True,2,0,PC (Microsoft Windows),True,72.0,Portal 2
4,11591,72,False,False,True,2,0,True,2,0,Xbox 360,True,72.0,Portal 2
5,1631,83,False,True,True,2,0,False,0,0,PlayStation 2,False,83.0,Baldur's Gate: Dark Alliance
6,17435,83,False,False,False,0,0,False,0,0,PC (Microsoft Windows),True,83.0,Baldur's Gate: Dark Alliance
7,11138,121,True,True,False,0,0,False,0,0,Mac,False,121.0,Minecraft: Java Edition
8,11137,121,True,True,False,0,0,False,0,0,PC (Microsoft Windows),False,121.0,Minecraft: Java Edition
9,11139,121,True,True,False,0,0,False,0,0,Linux,False,121.0,Minecraft: Java Edition


Duplicate IDS often seems to indicate multi-platform releases

In [44]:
full_df = modes_df.merge(games_df, left_on='game', right_on='id', how='left', suffixes=('', '_game'))
full_df.head()

,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,9953,92273,False,False,False,0,2,False,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1832,7153,True,True,True,2,0,False,0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7987,57887,False,False,True,2,2,False,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,7,46076,False,False,False,0,30,False,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10207,31256,False,False,False,0,0,False,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Handling conflicting values across columns

In [45]:
columns = ['offlinecoop', 'offlinecoopmax', 'offlinemax']

def cap_at_1(x):
    return 1 if x >= 1 else x

for col in columns[1:]:
    full_df[col] = full_df[col].apply(cap_at_1)

full_df[columns[-1]].value_counts()

offlinemax
0    16437
1     7435
Name: count, dtype: int64

In [46]:
#Sample one row from each unique combination of columns
selected_cols = ['name', 'first_release_date', 'platform'] + [col for col in full_df.columns if 'game_modes' in col]
conflict_scenarios_df = full_df.groupby(columns)[selected_cols].count()#agg(lambda x: x.sample(1))

#conflict_scenarios_df = conflict_scenarios_df.drop(labels = [()])
conflict_scenarios_df

name  first_release_date  platform  \
offlinecoop offlinecoopmax offlinemax                                        
False       0              0           14814               14814     14850   
                           1            4739                4739      4742   
True        1              0            1585                1585      1587   
                           1            2692                2692      2693   

                                       game_modes_Battle Royale  \
offlinecoop offlinecoopmax offlinemax                             
False       0              0                              14814   
                           1                               4739   
True        1              0                               1585   
                           1                               2692   

                                       game_modes_Co-operative  \
offlinecoop offlinecoopmax offlinemax                            
False       0              0                             14814   
                           1                              4739   
True        1              0                              1585   
                           1                              2692   

                                       game_modes_Massively Multiplayer Online (MMO)  \
offlinecoop offlinecoopmax offlinemax                                                  
False       0              0                                                   14814   
                           1                                                    4739   
True        1              0                                                    1585   
                           1                                                    2692   

                                       game_modes_Multiplayer  \
offlinecoop offlinecoopmax offlinemax                           
False       0              0                            14814   
                           1                             4739   
True        1              0                             1585   
                           1                             2692   

                                       game_modes_Single player  \
offlinecoop offlinecoopmax offlinemax                             
False       0              0                              14814   
                           1                               4739   
True        1              0                               1585   
                           1                               2692   

                                       game_modes_Split screen  
offlinecoop offlinecoopmax offlinemax                           
False       0              0                             14814  
                           1                              4739  
True        1              0                              1585  
                           1                              2692

After inspecting some examples of each combination of the `offlinecoop`, `offlinecoopmax`, and `offlinemax` columns; the meanings can be interpreted as follows:

| offlinecoop | offlinecoopmax | offlinemax | Meaning |
| -------- | ------- | -------- | -------- |
| True | -1 | 1 | Misinput; trust data from games endpoint |
| True | 0 | 1 | Used the other column, fill coop max with max |
| True | 1 | -1 | Offline coop but no PVP |
| True | 1 | 0 | Offline coop but no PVP |
| True | 1 | 1 | Offline coop and PVP |
| False | -1 | -1 | No offline play |
| False | -1 | 0 | No offline play |
| False | -1 | 1 | Offline PVP but no coop |
| False | 0 | -1 | No offline play |
| False | 0 | 0 | No offline play |
| False | 0 | 1 | Inconsistent, assume no offline play |
| False | 1 | -1 | Inconsistent; assume no offline play|
| False | 1 | 0 | Inconsistent; assume no offline play|


There are only 5 error cases:

1. offlinecoop = True, offlinecoopmax \< 0, and offlinemax \> 0 

2. offlinecoop = True, offlinecoopmax = 0, and offlinemax \> 0 

3. offlinecoop = False, offlinecoopmax = 0, and offlinemax \> 0 

4. offlinecoop = False, offlinecoopmax \> 0, and offlinemax \< 0 

5. offlinecoop = False, offlinecoopmax \> 0, and offlinemax = 0 


In [47]:
from data_cleaning_functions import get_replacement_function

offline = {'coop_column' : 'offlinecoop',
           'coop_max_column' : 'offlinecoopmax',
           'max_column' : 'offlinemax'}
online = {'coop_column' : 'onlinecoop',
           'coop_max_column' : 'onlinecoopmax',
           'max_column' : 'onlinemax'}

fixed_df = full_df.apply(get_replacement_function(**offline), axis = 1)
fixed_df = fixed_df.apply(get_replacement_function(**online), axis = 1)

conflict_scenarios_df = fixed_df.groupby(columns)[selected_cols].agg(lambda x: x.sample(1))

#conflict_scenarios_df = conflict_scenarios_df.drop(labels = [()])
conflict_scenarios_df

name  \
offlinecoop offlinecoopmax offlinemax                                              
False       0              0                                           Dreadzone   
True        1              0           The Dark Pictures Anthology: Man of Medan   
                           1                            Rocket League: Season 17   

                                      first_release_date  \
offlinecoop offlinecoopmax offlinemax                      
False       0              0                  2003-10-02   
True        1              0                  2017-07-31   
                           1                  2010-06-25   

                                                                  platform  \
offlinecoop offlinecoopmax offlinemax                                        
False       0              0                                        Arcade   
True        1              0                        PC (Microsoft Windows)   
                           1           Super Nintendo Entertainment System   

                                       game_modes_Battle Royale  \
offlinecoop offlinecoopmax offlinemax                             
False       0              0                                0.0   
True        1              0                                0.0   
                           1                                0.0   

                                       game_modes_Co-operative  \
offlinecoop offlinecoopmax offlinemax                            
False       0              0                               0.0   
True        1              0                               1.0   
                           1                               1.0   

                                       game_modes_Massively Multiplayer Online (MMO)  \
offlinecoop offlinecoopmax offlinemax                                                  
False       0              0                                                     0.0   
True        1              0                                                     0.0   
                           1                                                     0.0   

                                       game_modes_Multiplayer  \
offlinecoop offlinecoopmax offlinemax                           
False       0              0                              1.0   
True        1              0                              1.0   
                           1                              1.0   

                                       game_modes_Single player  \
offlinecoop offlinecoopmax offlinemax                             
False       0              0                                0.0   
True        1              0                                1.0   
                           1                                1.0   

                                       game_modes_Split screen  
offlinecoop offlinecoopmax offlinemax                           
False       0              0                               0.0  
True        1              0                               0.0  
                           1                               0.0

## Handling Outliers

In [48]:
onlinemax_column = modes_df['onlinemax'].sort_values(ascending = False)
outliers = onlinemax_column.iloc[:10]

In [49]:
max_game = modes_df.loc[outliers.index]
max_game

,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen
2540,9432,117526,True,False,False,0,0,True,100,100,Web browser,False
2537,8927,54734,False,False,False,0,0,False,0,100,PC (Microsoft Windows),False
13654,24854,272339,False,False,False,0,1,False,0,100,PC (Microsoft Windows),False
8568,16982,10239,True,False,False,0,0,True,100,100,Xbox One,True
18603,30472,264138,False,True,False,0,0,True,4,100,SteamVR,False
18581,30145,249829,False,False,False,0,0,True,4,100,PC (Microsoft Windows),False
12159,22336,217815,False,False,False,0,0,True,4,100,PC (Microsoft Windows),False
9393,18150,191536,True,True,False,0,0,True,100,100,Web browser,False
6549,14178,145566,True,False,False,0,0,True,100,100,PC (Microsoft Windows),False
18258,30361,317486,False,False,False,0,0,False,0,100,Android,False


In [50]:
max_game.merge(games_df, how = 'left', left_on = 'game', right_on = 'id')['name']

0                          Lurkers
1                  Escalation 1985
2                       EX-Xdriver
3            Ark: Survival Evolved
4                     Ascent Quest
5                            Flayl
6            Call of Duty: Warzone
7    Let's Go! Baby! Friends World
8               Moonlight Princess
9                         Aooni.io
Name: name, dtype: str

None of these games actually support over 100 players; we need to drop these outliers.

In [51]:
modes_df = modes_df.drop(index = outliers)